# TP2 — Reconstrucción 3D con cámara estéreo

**I308 Visión Artificial · UdeSA · Santiago Luis Groba Alonso**

Reconstrucción 3D de un objeto rígido (buda) a partir de pares de imágenes RGB capturados con la cámara estéreo ELP-USB3D1080P02-H120. Las celdas markdown describen, en pasado, lo que hace cada etapa — sirven tanto de guía para escribir el código como de borrador para el informe.

## 0. Setup

Importamos las librerias del pipeline (`cv2`, `numpy`, `matplotlib`, `open3d`, `tqdm`) y los modulos locales de catedra: `calib.py` (helpers de calibracion: `board_points`, `np_print`, `draw_checkerboard`) y `aruco.py` (helpers de ChArUco para la pose en escena). Definimos los paths del dataset (`datasets/stereo_propio/`).

In [ ]:
import glob
import os
import pickle
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
from tqdm import tqdm

# Helpers de catedra (copiados localmente desde ejemplos clase/)
import calib   # board_points, detect_board, np_print, draw_checkerboard
import aruco   # create_charuco_board, detect_charuco_markers, ...

# --- Dataset propio ---
DATASET_DIR = Path("datasets/stereo_propio")
CALIB_DIR = DATASET_DIR / "calib"
CAPTURES_DIR = DATASET_DIR / "captures"

# --- Chessboard de calibracion ---
# Hoja A4 con tablero 7x10 cuadrados liso, sin markers. Esquinas internas 6x9.
# Square size: 210 mm (ancho A4) / 7 cuadrados = 30 mm.
CHECKERBOARD = (6, 9)        # (cols, rows) de esquinas internas
SQUARE_SIZE_MM = 30.0

# --- ChArUco board para pose en escena (seccion 4) ---
# Diccionario raro (DICT_ARUCO_MIP_36H12) encontrado por fuerza bruta. Este
# board aparece en las capturas del buda y se usa para anclar el frame del
# mundo en cada toma -- NO se usa para calibrar intrinsecos.
CHARUCO_DICT = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_ARUCO_MIP_36H12)
CHARUCO_MARKER_MM = 64.0   # lado fisico del marker (mm)

print("Setup OK")

## 1. Calibracion estereo

El target de calibracion es un **chessboard liso 7×10 cuadrados** (sin markers ArUco) impreso en A4. Las esquinas internas son **6×9 = 54 puntos por imagen** y cada cuadrado mide 30 mm (210 mm / 7 cuadrados de ancho de A4).

`cv2.findChessboardCorners` (envuelto en `calib.detect_board` con `cornerSubPix`) encuentra las 54 esquinas como una vista coherente por imagen -- sin la ambiguedad planar que tienen los markers individuales. Cada par estereo donde se detecta en AMBAS camaras aporta una vista completa al solver.

**Pipeline**:
1. Detectar el chessboard en cada `left_i.jpg` y `right_i.jpg` (subpix refinement).
2. Acumular las vistas donde se detecta en ambas.
3. Pre-calibrar cada camara con `cv2.calibrateCamera` para obtener intrinsecos iniciales solidos.
4. `cv2.stereoCalibrate` con `CALIB_FIX_INTRINSIC` para estimar solo `R, T` entre camaras.
5. Serializar a `stereo_calibration.pkl`.

> **Nota**: el patron ChArUco chico que aparece en escena (junto al teclado en algunas calib images) NO se usa para calibracion -- es para estimar pose del mundo en cada captura del buda (seccion 4).

In [ ]:
# 1.1 Puntos 3D del chessboard en su propio frame
# calib.board_points devuelve un grid (cols * rows, 3) con coordenadas en
# unidades de cuadrado y Z=0. Multiplicamos por SQUARE_SIZE_MM para tener mm.
board_obj_points = (calib.board_points(CHECKERBOARD) * SQUARE_SIZE_MM).astype(np.float32)
print(f"Esquinas internas: {CHECKERBOARD[0]} x {CHECKERBOARD[1]} = {CHECKERBOARD[0]*CHECKERBOARD[1]} puntos / imagen")
print(f"Square size: {SQUARE_SIZE_MM} mm")
print(f"board_obj_points shape: {board_obj_points.shape}")

In [ ]:
# 1.2 Listar pares de imagenes de calibracion (sorted numericamente)
def numeric_sort(file_name):
    return int(os.path.basename(file_name).split("_")[-1].split(".")[0])

left_file_names = sorted(
    glob.glob(str(CALIB_DIR / "*left*.jpg")),
    key=numeric_sort,
)
right_file_names = sorted(
    glob.glob(str(CALIB_DIR / "*right*.jpg")),
    key=numeric_sort,
)
assert len(left_file_names) == len(right_file_names)
print(f"Pares de calibracion en {CALIB_DIR}: {len(left_file_names)}")

In [ ]:
# 1.3 Loop: detectar chessboard en L y R, paralelizado con ThreadPoolExecutor
# (cv2.findChessboardCorners libera el GIL -> threads dan paralelismo real,
# ~10x con 12 cores). Acumulamos vistas donde se detecta en AMBAS camaras.
from concurrent.futures import ThreadPoolExecutor

def detect_pair(left_fn, right_fn):
    left_image = cv2.imread(left_fn, cv2.IMREAD_GRAYSCALE)
    right_image = cv2.imread(right_fn, cv2.IMREAD_GRAYSCALE)
    sz = (left_image.shape[1], left_image.shape[0])
    ret_L, corners_L = calib.detect_board(CHECKERBOARD, left_image)
    ret_R, corners_R = calib.detect_board(CHECKERBOARD, right_image)
    return sz, ret_L, corners_L, ret_R, corners_R

with ThreadPoolExecutor(max_workers=12) as ex:
    results = list(tqdm(
        ex.map(detect_pair, left_file_names, right_file_names),
        total=len(left_file_names), desc="Detectando chessboard",
    ))

object_points = []
left_images_points = []
right_images_points = []
image_size = None
for sz, ret_L, corners_L, ret_R, corners_R in results:
    if image_size is None:
        image_size = sz
    if not (ret_L and ret_R):
        continue
    object_points.append(board_obj_points)
    left_images_points.append(corners_L)
    right_images_points.append(corners_R)

n_pairs_ok = len(object_points)
print(f"\nPares con chessboard detectado en L y R: {n_pairs_ok} / {len(left_file_names)}")
print(f"image_size = {image_size}")
assert n_pairs_ok >= 15, "Muy pocas vistas, calibracion va a ser mala"

In [ ]:
# 1.4 Calibracion estereo desacoplada: primero cada camara por separado, despues
# stereoCalibrate con CALIB_FIX_INTRINSIC.
# Subsampleamos a MAX_VIEWS evenly-spaced: con >40 vistas el LM se vuelve lento
# y el marginal de informacion es chico (la calibracion satura ~30-40 vistas
# bien distribuidas).
MAX_VIEWS = 40
if n_pairs_ok > MAX_VIEWS:
    idx = np.linspace(0, n_pairs_ok - 1, MAX_VIEWS).astype(int)
    obj_sub  = [object_points[i] for i in idx]
    left_sub = [left_images_points[i] for i in idx]
    right_sub = [right_images_points[i] for i in idx]
    print(f"Sub-sampleando {n_pairs_ok} -> {MAX_VIEWS} vistas para acelerar LM")
else:
    obj_sub, left_sub, right_sub = object_points, left_images_points, right_images_points

n_views = len(obj_sub)

print(f"Pre-calibrando camara IZQUIERDA sobre {n_views} vistas...")
err_L, K_L_init, D_L_init, _, _ = cv2.calibrateCamera(
    obj_sub, left_sub, image_size, None, None,
)
print(f"  RMS izq: {err_L:.3f} px")

print(f"Pre-calibrando camara DERECHA...")
err_R, K_R_init, D_R_init, _, _ = cv2.calibrateCamera(
    obj_sub, right_sub, image_size, None, None,
)
print(f"  RMS der: {err_R:.3f} px")

print(f"\nStereoCalibrate con CALIB_FIX_INTRINSIC...")
err, left_K, left_dist, right_K, right_dist, R, T, E, F = cv2.stereoCalibrate(
    obj_sub, left_sub, right_sub,
    K_L_init, D_L_init, K_R_init, D_R_init,
    image_size,
    flags=cv2.CALIB_FIX_INTRINSIC,
)
print(f"RMS re-projection error: {err:.3f} px  (<1 px excelente, <2 px aceptable)")
print(f"baseline ||T||: {np.linalg.norm(T):.2f} mm  (fabricante ELP-USB3D1080P02-H120 ~60 mm)")

In [ ]:
# 1.5 Imprimir resultados con np_print (formato lindo de catedra)
to_print = [
    "# Left camera intrinsics:",
    ("left_K",    left_K),
    ("left_dist", left_dist),
    "# Right camera intrinsics:",
    ("right_K",    right_K),
    ("right_dist", right_dist),
    "# Rotation L -> R:",
    ("R", R),
    "# Translation L -> R (mm):",
    ("T", T),
]
for entry in to_print:
    if isinstance(entry, str):
        print(entry)
    else:
        name, arr = entry
        print(f"{name} = {calib.np_print(arr)}\n")

In [ ]:
# 1.6 Serializar la calibracion a pkl (igual que catedra stereo.ipynb cell 21)
calibration_results = {
    'left_K': left_K, 'left_dist': left_dist,
    'right_K': right_K, 'right_dist': right_dist,
    'R': R, 'T': T, 'E': E, 'F': F,
    'image_size': image_size,
    'rms_error': float(err),
    'n_pairs_ok': n_pairs_ok,
}

calibration_file = DATASET_DIR / "stereo_calibration.pkl"
with open(calibration_file, "wb") as f:
    pickle.dump(calibration_results, f)
print(f"Guardado en {calibration_file}")

# Alias para celdas siguientes
K_L, D_L = left_K, left_dist
K_R, D_R = right_K, right_dist
IMAGE_SIZE = image_size
baseline_mm = float(np.linalg.norm(T))

## 2. Rectificacion estereo

A partir de la calibracion (`K_L`, `D_L`, `K_R`, `D_R`, `R`, `T`), `cv2.stereoRectify` calcula las matrices de proyeccion `P1`, `P2` y la matriz de reproyeccion `Q` (que mapea `(u, v, disparidad)` a `(X, Y, Z)` en el frame de camara rectificada).

`cv2.initUndistortRectifyMap` genera los mapas que `cv2.remap` usa en runtime para aplicar de una sola pasada des-distorsion + rectificacion. Usamos `cv2.CV_32FC1` (mapas en float, como el ejemplo de catedra en stereo.ipynb cell 27).

In [ ]:
# 2.1 cv2.stereoRectify (catedra stereo.ipynb cell 25)
# alpha=0 recorta el negro de los bordes (mejor para SGBM); =1 conserva todo
R1, R2, P1, P2, Q, validRoi1, validRoi2 = cv2.stereoRectify(
    left_K, left_dist, right_K, right_dist, image_size, R, T, alpha=0,
)
print(f"P1 =\n{P1}\n")
print(f"P2 =\n{P2}\n")
print(f"Q  =\n{Q}\n")
print(f"validRoi1 = {validRoi1}")
print(f"validRoi2 = {validRoi2}")
print(f"baseline efectivo (-P2[0,3]/P2[0,0]): {-P2[0,3]/P2[0,0]:.2f} mm")

In [ ]:
# 2.2 Mapas de des-distorsion + rectificacion (catedra stereo.ipynb cell 27)
left_map_x, left_map_y = cv2.initUndistortRectifyMap(
    left_K, left_dist, R1, P1, image_size, cv2.CV_32FC1,
)
right_map_x, right_map_y = cv2.initUndistortRectifyMap(
    right_K, right_dist, R2, P2, image_size, cv2.CV_32FC1,
)
print(f"map shape: {left_map_x.shape}  (igual a image_size {image_size})")
print(f"map dtype: {left_map_x.dtype}")

# Helper para rectificar pares (lo usamos despues en el loop de fusion)
def rectify(left, right):
    left_rect = cv2.remap(left, left_map_x, left_map_y, cv2.INTER_LINEAR)
    right_rect = cv2.remap(right, right_map_x, right_map_y, cv2.INTER_LINEAR)
    return left_rect, right_rect

In [ ]:
# 2.3 Aplicar a un par de capturas para verificar que las epipolares quedan
# horizontales. Mismo helper que catedra (stereo.ipynb cell 38).
left_raw = cv2.imread(str(CAPTURES_DIR / "left_0.jpg"))
right_raw = cv2.imread(str(CAPTURES_DIR / "right_0.jpg"))
left_rect, right_rect = rectify(left_raw, right_raw)

# Mostrar lado a lado con lineas horizontales para sanity-check de epipolares
fig, axes = plt.subplots(2, 1, figsize=(16, 11))
for ax, (Lp, Rp, t) in zip(axes, [
    (left_raw, right_raw, "ORIGINAL: epipolares NO horizontales"),
    (left_rect, right_rect, "RECTIFICADO: epipolares paralelas a las filas"),
]):
    combined = np.hstack([cv2.cvtColor(Lp, cv2.COLOR_BGR2RGB),
                          cv2.cvtColor(Rp, cv2.COLOR_BGR2RGB)])
    ax.imshow(combined)
    h = combined.shape[0]
    for y in np.linspace(0, h-1, 22)[1:-1]:
        ax.axhline(y=y, color="red", linewidth=0.5, alpha=0.7)
    ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Disparidad

Disparidad = diferencia horizontal entre la posicion de un mismo punto en las imagenes left y right rectificadas. A mas disparidad, mas cerca de la camara: `Z = f * B / d`.

Catedra muestra tres metodos en [stereo.ipynb](ejemplos%20clase/stereo.ipynb):

- **`cv2.StereoBM_create`**: Block Matching clasico. Rapido, ruidoso, sufre con zonas sin textura.
- **`cv2.StereoSGBM_create`**: Semi-Global Block Matching. Mejor cobertura, mas costoso.
- **CREStereo** (deep learning, CVPR 2022): red recurrente con correlacion adaptativa. Disparidad densa con bordes limpios y resistencia a zonas planas. Es el que usa catedra para el output final del pipeline.

Elegimos **CREStereo** via la libreria `disparity` que provee catedra en `ejemplos clase/disparity/`. Corre ONNX -- requiere `onnxruntime`. El modelo se descarga la primera vez a `models/` y queda cacheado.

Salida en pixels (float32); para validez usamos `disparity > 0`.

In [ ]:
# 3.1 Disparidad con CREStereo (CRE Stereo, CVPR 2022).
# Setup tomado de catedra (stereo.ipynb): Calibration + InputPair + method.compute_disparity.
# La primera ejecucion descarga el modelo .onnx (~30 MB) a models/; despues queda cacheado.
import sys
sys.path.insert(0, "ejemplos clase")
from disparity.method_cre_stereo import CREStereo
from disparity.methods import Calibration, InputPair, Config

w, h = IMAGE_SIZE
fx, fy = float(K_L[0, 0]), float(K_L[1, 1])
cx0, cy0 = float(K_L[0, 2]), float(K_L[1, 2])

calibration = Calibration(**{
    "width": w, "height": h,
    "baseline_meters": baseline_mm / 1000,
    "fx": fx, "fy": fy,
    "cx0": cx0, "cx1": cx0, "cy": cy0,
    "depth_range": [0.05, 20.0],
    "left_image_rect_normalized": [0, 0, 1, 1],
})

models_path = Path("models"); models_path.mkdir(exist_ok=True)
config = Config(models_path=models_path)

method = CREStereo(config)
method.parameters["Shape"].set_value("1280x720")  # 320x240 / 640x480 / 1280x720
# Defaults: Mode="combined" (2 pasadas), Iterations=10. Mas iters = mejor, mas lento.

def compute_disparity(left_rect, right_rect):
    pair = InputPair(left_rect, right_rect, calibration)
    return method.compute_disparity(pair).disparity_pixels


disparity_map = compute_disparity(left_rect, right_rect)
valid = disparity_map > 0
print(f"disparidad   min={disparity_map[valid].min():.2f}, max={disparity_map[valid].max():.2f}, mean={disparity_map[valid].mean():.2f} px")
print(f"cobertura    {valid.mean()*100:.1f}% de los pixeles validos")

f_px = P1[0, 0]
B_mm = -P2[0, 3] / P2[0, 0]
z_min = f_px * B_mm / disparity_map[valid].max()
z_max = f_px * B_mm / disparity_map[valid].min()
print(f"profundidad  ~{z_min:.0f} mm a ~{z_max:.0f} mm")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(left_rect, cv2.COLOR_BGR2RGB))
axes[0].set_title("Izquierda rectificada"); axes[0].axis("off")
disp_show = np.where(valid, disparity_map, np.nan)
vmin = float(np.nanpercentile(disp_show, 2))
vmax = float(np.nanpercentile(disp_show, 98))
im = axes[1].imshow(disp_show, cmap="plasma", vmin=vmin, vmax=vmax)
axes[1].set_title("Disparidad CREStereo")
axes[1].axis("off")
fig.colorbar(im, ax=axes[1], fraction=0.04, label="px")
plt.tight_layout(); plt.show()

## 4. Pose de la cámara respecto al mundo

Para fusionar nubes parciales en un único marco de coordenadas necesitamos saber, para cada par, cómo está orientada la cámara izquierda respecto al mundo. Definimos ese mundo anclando el origen al ChArUco que aparece en cada captura. Creamos el `CharucoBoard` con `aruco.create_charuco_board`, lo detectamos en la imagen izquierda **original** (no la rectificada — los intrínsecos `K_L`, `D_L` describen la imagen sin rectificar) y estimamos la pose con la helper `estimate_camera_pose_with_homography` que resuelve PnP. El resultado (`rvec`, `tvec`) lo convertimos a una matriz homogénea 4×4 `T_wc` con `cv2.Rodrigues`. Para sanity-check proyectamos los ejes del mundo sobre la imagen y los dibujamos en RGB.

## 5. Reconstrucción 3D de un par

Combinando el mapa de disparidad con la matriz `Q`, `cv2.reprojectImageTo3D` nos devuelve un array `(H, W, 3)` con la posición 3D de cada píxel en el frame de la cámara izquierda. Filtramos los píxeles con disparidad ≤ 0 (inválidos) y los puntos fuera del rango razonable de profundidad (100-2000 mm en nuestro setup, suficiente para enmarcar el buda sin traer fondo). Para llevar esos puntos al frame del mundo invertimos `T_wc` analíticamente (`T_cw = [R^T | -R^T·t]`) y aplicamos la transformación homogénea al batch entero. Armamos una `o3d.geometry.PointCloud` con esos puntos y les pegamos los colores del píxel correspondiente en la izquierda rectificada.

## 6. Loop sobre todos los pares + acumulación

Iteramos sobre todos los `(left_i.jpg, right_i.jpg)` del directorio `captures/`, repitiendo para cada par la secuencia rectificar → disparidad → detectar ChArUco → estimar pose → reproyectar a 3D → transformar al mundo. Sobre cada nube parcial aplicamos tres filtros: recorte con un `AxisAlignedBoundingBox` definido en coordenadas de mundo (descarta el plano del patrón y todo lo que esté fuera del bounding box del buda), `remove_statistical_outlier` con 20 vecinos y σ=2 para sacar puntos espurios del matching, y `voxel_down_sample` con voxel de 1 mm para mantener la densidad manejable. Acumulamos sumando las point clouds con `+=`. Llevamos un registro de los pares donde la detección del ChArUco o el solvePnP fallaron y los reportamos al final.

## 7. Exportar `.PLY` y estimar el alto del objeto

Guardamos la nube final con `o3d.io.write_point_cloud(..., write_ascii=False)` para que el archivo quede en binario y pese menos que el formato ASCII de PLY. Para estimar el alto del buda en milímetros calculamos la extensión de la nube sobre el eje vertical del mundo — en nuestro setup ese eje es `Z`, porque el ChArUco está apoyado sobre el plano `XY`. Reportamos también las extensiones sobre `X` e `Y` como sanity-check de que la escala métrica de la reconstrucción tenga sentido, y comparamos el alto estimado contra una medición manual del objeto con regla.